# AI Agents Workshop — Day 1 Labs (Gemini Edition)

S4DS KJSIT. Run the setup cell first, then work down.

**Before anything:** set `GEMINI_API_KEY` in your environment, or paste it when prompted.
Use `[GEMINI_API_KEY]` as the placeholder value in shared notebooks and avoid hardcoding real credentials.

This notebook mirrors the original HF labs, but migrates all model calls to Google Gemini (`google-generativeai`).

## Setup — run this once

In [2]:
!pip install -q google-generativeai requests duckduckgo-search

import os
from getpass import getpass

import google.generativeai as genai

# HF migration note: InferenceClient(...) -> genai.GenerativeModel(...)
api_key = os.environ.get("GEMINI_API_KEY", "[GEMINI_API_KEY]").strip()
if not api_key or api_key == "[GEMINI_API_KEY]":
    entered = getpass("GEMINI_API_KEY not found. Enter key (leave blank to keep placeholder): ")
    if entered.strip():
        api_key = entered.strip()

os.environ["GEMINI_API_KEY"] = api_key
MODEL_ID = "gemini-3.5-flash-lite"

genai.configure(api_key=os.environ["GEMINI_API_KEY"])
model = genai.GenerativeModel(MODEL_ID)

try:
    r = model.generate_content("Reply with exactly: pong")
    text = (r.text or "").strip()
    print("model said:", text)
    print("SETUP OK" if "pong" in text.lower() else "SETUP CHECK: unexpected output")
except Exception as exc:
    print(f"SETUP FAILED: {exc}")
    print("If this is rate limit/network related, retry in a minute and verify your API key.")

---
## Lab 1 — What the model actually sees

An LLM does not see a list of messages. It sees **structured content** serialized by the client/runtime.

In [ ]:
messages = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user", "content": "What is the capital of Maharashtra?"},
    {"role": "assistant", "content": "Mumbai."},
    {"role": "user", "content": "And its population?"},
]

# HF migration note: tokenizer.apply_chat_template(...) -> explicit message serialization
def render_chat_for_gemini(msgs):
    lines = []
    for m in msgs:
        lines.append(f"<{m['role']}>\n{m['content']}")
    lines.append("<assistant>")
    return "\n\n".join(lines)

prompt = render_chat_for_gemini(messages)
print(repr(prompt))
print()
print(prompt)

**Try it:** swap `MODEL_ID` to `gemini-1.5-pro`. Different models may respond differently even with the same serialized context.
This is why prompts do not transfer perfectly across models/providers.

---
## Lab 2 — A tool is a function + a description

In [ ]:
import inspect

def get_weather(city: str) -> str:
    """Get the current weather for an Indian city.

    Args:
        city: Name of the city, e.g. "Pune".
    """
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

def describe(func):
    sig = inspect.signature(func)
    doc = (func.__doc__ or "").strip().split("\n")[0]
    return f"- {func.__name__}{sig}: {doc}"

print(describe(get_weather))
print()
print("^ THIS is the only thing the model ever sees about your function.")

**Try it:** delete the docstring and re-run. Your docstring *is* your prompt.

---
## Lab 2.5 — Why bother with tools at all?

A tool is only worth its complexity if the model cannot already answer reliably.
So let us prove it: same question, asked twice.

1. **No tool** — the model answers from memory.
2. **With a tool** — we search first and paste results into the prompt.

Watch what changes.

In [ ]:
# Round 1 - no tool. The model answers from memory.

QUESTION = "What is the latest stable version of Python, and when was it released?"

def ask(prompt, max_tokens=250):
    # HF migration note: chat.completions.create(...) -> generate_content(...)
    try:
        r = model.generate_content(
            prompt,
            generation_config={"max_output_tokens": max_tokens, "temperature": 0.2},
        )
        return (r.text or "").strip()
    except Exception as exc:
        return f"[Gemini API error] {exc}"

no_tool = ask(QUESTION)

print("QUESTION:", QUESTION)
print("\n--- NO TOOL (from memory) ---")
print(no_tool)

---
## Lab 3 — Write the agent loop yourself

The most important cell in this notebook. Read it line by line before running.

In [ ]:
import re

MAX_STEPS = 6

def get_weather(city: str) -> str:
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: only numbers and + - * / ( ) allowed."
    try:
        return str(eval(expression))
    except Exception as exc:
        return f"Error: {exc}"

TOOLS = {"get_weather": get_weather, "calculate": calculate}

SYSTEM_PROMPT = """You solve tasks by reasoning step by step and using tools.

Available tools:
- get_weather(city): current weather for an Indian city.
- calculate(expression): evaluate arithmetic.

Reply in exactly this format, one step at a time:

Thought: <your reasoning>
Action: <tool_name>(<single argument>)

After each Action you will be shown an Observation.
When done, reply with:

Thought: <why you can answer now>
Final Answer: <your answer>

Never write an Observation yourself."""

ACTION_RE = re.compile(r"Action:\s*(\w+)\((.*?)\)", re.DOTALL)

def format_messages(messages):
    lines = []
    for m in messages:
        lines.append(f"{m['role'].upper()}: {m['content']}")
    return "\n\n".join(lines)

def gemini_step(messages, max_tokens=400):
    prompt = format_messages(messages)
    try:
        resp = model.generate_content(
            prompt,
            generation_config={"max_output_tokens": max_tokens, "temperature": 0.2},
        )
        return (resp.text or "").strip()
    except Exception as exc:
        return f"Thought: API failure.\nFinal Answer: Gemini request failed with error: {exc}"

def run(task):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, MAX_STEPS + 1):
        print(f"\n{'-'*50}\nSTEP {step}\n{'-'*50}")
        out = gemini_step(messages, max_tokens=400)
        print(out)
        messages.append({"role": "assistant", "content": out})

        if "Final Answer:" in out:
            return out.split("Final Answer:", 1)[1].strip()

        m = ACTION_RE.search(out)
        if not m:
            messages.append({
                "role": "user",
                "content": "Invalid format. Use 'Action: tool(arg)' or 'Final Answer: ...'."
            })
            continue

        name, arg = m.group(1), m.group(2).strip().strip("\"'")
        obs = TOOLS[name](arg) if name in TOOLS else f"Error: no tool '{name}'"
        print(f"\nObservation: {obs}")
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "Gave up - hit MAX_STEPS."

print(run("What's the weather in Pune, and what is that temperature plus 5?"))

**Try it:** remove the strict format checks and re-run. You will usually get inconsistent tool behavior.
Guardrails in the loop are what keep the agent grounded.

---
## Lab 4 — Native Gemini function calling

Same high-level idea as framework agents: model plans, tools execute, model synthesizes final answer.

In [ ]:
import json

def get_weather(city: str) -> str:
    """Get the current weather for an Indian city.

    Use this whenever the user asks about temperature, rain, or humidity.

    Args:
        city: Name of the city, e.g. "Pune".
    """
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

def get_mess_menu(day: str) -> str:
    """Get the hostel mess menu for a day of the week.

    Args:
        day: Day name, e.g. "Tuesday".
    """
    menu = {"monday": "Rajma chawal", "tuesday": "Pav bhaji", "wednesday": "Veg biryani"}
    return menu.get(day.lower().strip(), f"No menu for {day}")

tool_map = {"get_weather": get_weather, "get_mess_menu": get_mess_menu}

tools = [{
    "function_declarations": [
        {
            "name": "get_weather",
            "description": "Get the current weather for an Indian city.",
            "parameters": {
                "type": "OBJECT",
                "properties": {"city": {"type": "STRING"}},
                "required": ["city"],
            },
        },
        {
            "name": "get_mess_menu",
            "description": "Get the hostel mess menu for a day of the week.",
            "parameters": {
                "type": "OBJECT",
                "properties": {"day": {"type": "STRING"}},
                "required": ["day"],
            },
        },
    ]
}]

agent_model = genai.GenerativeModel(model_name=MODEL_ID, tools=tools)
query = "Weather in Pune and Tuesday's mess menu - good day to eat outside?"

try:
    chat = agent_model.start_chat()
    first = chat.send_message(query)

    tool_results = []
    for part in first.candidates[0].content.parts:
        fc = getattr(part, "function_call", None)
        if not fc:
            continue

        fn_name = fc.name
        args = dict(fc.args) if fc.args else {}

        if fn_name in tool_map:
            arg_value = args.get("city") if fn_name == "get_weather" else args.get("day")
            result = tool_map[fn_name](str(arg_value))
        else:
            result = f"Error: unknown tool {fn_name}"

        tool_results.append({"name": fn_name, "args": args, "result": result})

    if tool_results:
        summary = "\n".join([f"{t['name']}({t['args']}) -> {t['result']}" for t in tool_results])
        final_prompt = (
            "Tool outputs are ready. Write a concise final answer for the user.\n\n"
            f"User query: {query}\n"
            f"Tool outputs:\n{summary}"
        )
        final_resp = model.generate_content(final_prompt)
        print((final_resp.text or "").strip())
    else:
        print((first.text or "").strip())
except Exception as exc:
    print(f"Gemini function-calling flow error: {exc}")
    print("If this is temporary (quota/network), retry this cell.")

**Try it:** print the tool call arguments and intermediate outputs to inspect what Gemini planned before the final answer.
That is the easiest way to debug tool use quality.